In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Load the dataset
data = pd.read_csv("C:/Users/Altaf/Documents/ProposalStuff/from vbox/conn_combined_hoic.csv") 

# Preprocess
data_clean = data.drop(columns=['ts', 'uid', 'id.orig_h', 'id.resp_h'])
data_clean.replace('-', pd.NA, inplace=True)
data_clean.fillna(data_clean.mode().iloc[0], inplace=True)

# Encode the target variable (Label)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data_clean['Label'])  # Fit and transform on the target (Label)
X = data_clean.drop(columns=['Label'])

# Initialize the LabelEncoder for each categorical column
categorical_columns = ['proto', 'service', 'local_orig', 'local_resp', 'conn_state', 'history', 'tunnel_parents']
feature_encoders = {}  # Store the encoders for each feature

# Fit LabelEncoder for each categorical column in the training data
for col in categorical_columns:
    feature_encoders[col] = LabelEncoder()
    X[col] = feature_encoders[col].fit_transform(X[col])  # Fit and transform on training data

# Convert relevant columns to numeric (coerce errors to NaN)
X['duration'] = pd.to_numeric(X['duration'], errors='coerce')
X['orig_bytes'] = pd.to_numeric(X['orig_bytes'], errors='coerce')
X['resp_bytes'] = pd.to_numeric(X['resp_bytes'], errors='coerce')

# Impute any remaining NaN values (if any)
X.fillna(X.mode().iloc[0], inplace=True)

# Split the dataset into training and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Random Forest model
rf_model_loic = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model_loic.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_model_loic.predict(X_test)

# Evaluate the model's performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


C:\Users\Altaf\AppData\Local\Temp\ipykernel_21968\660266167.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_clean.fillna(data_clean.mode().iloc[0], inplace=True)


Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1170
           1       1.00      1.00      1.00     14351

    accuracy                           1.00     15521
   macro avg       1.00      1.00      1.00     15521
weighted avg       1.00      1.00      1.00     15521



In [2]:
new_data = pd.read_csv("C:/Users/Altaf/Documents/ProposalStuff/from vbox/conn_loic.csv")

# Preprocess the new data (similar to the preprocessing of the training data)
new_data_clean = new_data.drop(columns=['ts', 'uid', 'id.orig_h', 'id.resp_h'])
new_data_clean.replace('-', pd.NA, inplace=True)
new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)

# For each categorical column, handle unseen labels by mapping them to ANOMALY (-1)
for col in categorical_columns:
    # Identify unseen labels (labels not seen during training)
    unseen_labels = ~new_data_clean[col].isin(feature_encoders[col].classes_)
    
    if unseen_labels.any():
        # Treat unseen labels as "ANOMALY"
        new_data_clean.loc[unseen_labels, col] = -1  # Assign unseen values to -1
    
    # LabelEncoder's transformation to the valid (non-unseen) values
    valid_labels = ~new_data_clean[col].isin([-1]) 
    new_data_clean.loc[valid_labels, col] = feature_encoders[col].transform(new_data_clean.loc[valid_labels, col])

# Convert relevant columns to numeric (coerce errors to NaN)
new_data_clean['duration'] = pd.to_numeric(new_data_clean['duration'], errors='coerce')
new_data_clean['orig_bytes'] = pd.to_numeric(new_data_clean['orig_bytes'], errors='coerce')
new_data_clean['resp_bytes'] = pd.to_numeric(new_data_clean['resp_bytes'], errors='coerce')

# Impute any remaining NaN values (if any)
new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)

# Make predictions on the new data using the trained Random Forest model
predictions = rf_model_loic.predict(new_data_clean)

# Convert numeric predictions back to original labels using the LabelEncoder for the target
predicted_labels = label_encoder.inverse_transform(predictions)

# Print or save the predictions
print(predicted_labels)
pd.Series(predicted_labels).value_counts()

['ANOMALY' 'ANOMALY' 'ANOMALY' ... 'ANOMALY' 'ANOMALY' 'ANOMALY']


C:\Users\Altaf\AppData\Local\Temp\ipykernel_21968\3388405411.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)
C:\Users\Altaf\AppData\Local\Temp\ipykernel_21968\3388405411.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)


ANOMALY    31403
BENIGN      2157
Name: count, dtype: int64

In [3]:
new_data = pd.read_csv("C:/Users/Altaf/Documents/ProposalStuff/from vbox/conn_slow.csv")

# Preprocess the new data (similar to the preprocessing of the training data)
new_data_clean = new_data.drop(columns=['ts', 'uid', 'id.orig_h', 'id.resp_h'])
new_data_clean.replace('-', pd.NA, inplace=True)
new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)

# For each categorical column, handle unseen labels by mapping them to ANOMALY (-1)
for col in categorical_columns:
    # Identify unseen labels (labels not seen during training)
    unseen_labels = ~new_data_clean[col].isin(feature_encoders[col].classes_)
    
    if unseen_labels.any():
        # Treat unseen labels as "ANOMALY"
        new_data_clean.loc[unseen_labels, col] = -1  # Assign unseen values to -1
    
    # LabelEncoder's transformation to the valid (non-unseen) values
    valid_labels = ~new_data_clean[col].isin([-1]) 
    new_data_clean.loc[valid_labels, col] = feature_encoders[col].transform(new_data_clean.loc[valid_labels, col])

# Convert relevant columns to numeric (coerce errors to NaN)
new_data_clean['duration'] = pd.to_numeric(new_data_clean['duration'], errors='coerce')
new_data_clean['orig_bytes'] = pd.to_numeric(new_data_clean['orig_bytes'], errors='coerce')
new_data_clean['resp_bytes'] = pd.to_numeric(new_data_clean['resp_bytes'], errors='coerce')

# Impute any remaining NaN values (if any)
new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)

# Make predictions on the new data using the trained Random Forest model
predictions = rf_model_loic.predict(new_data_clean)

# Convert numeric predictions back to original labels using the LabelEncoder for the target
predicted_labels = label_encoder.inverse_transform(predictions)

# Print or save the predictions
print(predicted_labels)
pd.Series(predicted_labels).value_counts()

['BENIGN' 'BENIGN' 'BENIGN' ... 'ANOMALY' 'ANOMALY' 'ANOMALY']


C:\Users\Altaf\AppData\Local\Temp\ipykernel_21968\683941148.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)
C:\Users\Altaf\AppData\Local\Temp\ipykernel_21968\683941148.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)


ANOMALY    22982
BENIGN        42
Name: count, dtype: int64

In [4]:
new_data = pd.read_csv("C:/Users/Altaf/Documents/ProposalStuff/from vbox/conn_golden.csv")

# Preprocess the new data (similar to the preprocessing of the training data)
new_data_clean = new_data.drop(columns=['ts', 'uid', 'id.orig_h', 'id.resp_h'])
new_data_clean.replace('-', pd.NA, inplace=True)
new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)

# For each categorical column, handle unseen labels by mapping them to ANOMALY (-1)
for col in categorical_columns:
    # Identify unseen labels (labels not seen during training)
    unseen_labels = ~new_data_clean[col].isin(feature_encoders[col].classes_)
    
    if unseen_labels.any():
        # Treat unseen labels as "ANOMALY"
        new_data_clean.loc[unseen_labels, col] = -1  # Assign unseen values to -1
    
    # LabelEncoder's transformation to the valid (non-unseen) values
    valid_labels = ~new_data_clean[col].isin([-1]) 
    new_data_clean.loc[valid_labels, col] = feature_encoders[col].transform(new_data_clean.loc[valid_labels, col])

# Convert relevant columns to numeric (coerce errors to NaN)
new_data_clean['duration'] = pd.to_numeric(new_data_clean['duration'], errors='coerce')
new_data_clean['orig_bytes'] = pd.to_numeric(new_data_clean['orig_bytes'], errors='coerce')
new_data_clean['resp_bytes'] = pd.to_numeric(new_data_clean['resp_bytes'], errors='coerce')

# Impute any remaining NaN values (if any)
new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)

# Make predictions on the new data using the trained Random Forest model
predictions = rf_model_loic.predict(new_data_clean)

# Convert numeric predictions back to original labels using the LabelEncoder for the target
predicted_labels = label_encoder.inverse_transform(predictions)

# Print or save the predictions
print(predicted_labels)
pd.Series(predicted_labels).value_counts()

['ANOMALY' 'ANOMALY' 'ANOMALY' ... 'ANOMALY' 'ANOMALY' 'ANOMALY']


C:\Users\Altaf\AppData\Local\Temp\ipykernel_21968\1237580912.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)
C:\Users\Altaf\AppData\Local\Temp\ipykernel_21968\1237580912.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_data_clean.fillna(new_data_clean.mode().iloc[0], inplace=True)


ANOMALY    61573
Name: count, dtype: int64